# Task 04 — Multi-Turn Storage Ring Injection Validation Notebook

This notebook loads the storage ring lattice (`storage_ring_lattice_nkm.mat`), simulates multi-turn injection across four kicker models (NKM Off, Ideal Kicker, Linearized NKM, RADIA Fieldmap NKM), and evaluates physical multi-turn capture efficiency, loss accounting, and stored-beam perturbation.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Ensure repository root is on sys.path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.nkm_injection.storage_ring_injection import (
    StorageRingInjectionConfig,
    load_storage_ring_injection_lattice,
    track_multiturn_injection,
    compute_multiturn_injection_metrics
)
from src.nkm_injection.beam import generate_6d_beam
from src.nkm_injection.kickmap import NKMKickMap2D

In [ ]:
# ── Simulation Configuration Summary ─────────────────────────────────────────
# Prints a table of all simulation parameters: their defaults (from config
# dataclasses) and any values reconfigured explicitly in this notebook.

import sys, os
from pathlib import Path

# Ensure repo root is on sys.path (already done in Cell 1; kept for safety)
_repo_root = Path('..').resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.nkm_injection.storage_ring_injection import StorageRingInjectionConfig
from src.nkm_injection.beam import generate_6d_beam

# ── Pull defaults from config dataclass ──────────────────────────────────────
_default_cfg = StorageRingInjectionConfig()

# ── Values explicitly set in this notebook ───────────────────────────────────
_notebook_n_particles  = 100
_notebook_n_turns      = 10
_notebook_inj_beta_x   = 10.0   # m   (injected beam)
_notebook_inj_beta_y   = 5.0    # m
_notebook_inj_emit_x   = 1e-7   # m·rad
_notebook_inj_emit_y   = 1e-8   # m·rad
_notebook_inj_x_offset = -0.016 # m   (septum position)
_notebook_sto_beta_x   = 10.0   # m   (stored beam)
_notebook_sto_beta_y   = 5.0    # m
_notebook_sto_emit_x   = 1e-8   # m·rad
_notebook_sto_emit_y   = 1e-9   # m·rad
_notebook_sto_x_offset = 0.0    # m
_notebook_seed         = 42
_notebook_kicker_models = ['off', 'ideal', 'linear', 'fieldmap']

# ── Table printer ─────────────────────────────────────────────────────────────
def _changed(default, notebook):
    return '⟵ reconfigured' if default != notebook else ''

rows = [
    # header
    ('Parameter', 'Default', 'This Notebook', 'Unit', 'Note'),
    # separator
    ('-' * 35, '-' * 18, '-' * 18, '-' * 10, '-' * 20),
    # Lattice / source data
    ('mat_filename',
     _default_cfg.mat_filename, _default_cfg.mat_filename, '—', ''),
    # Tracking
    ('n_particles',
     '(user defined)', _notebook_n_particles, 'count',
     _changed('(user defined)', _notebook_n_particles)),
    ('n_turns',
     10, _notebook_n_turns, 'turns',
     _changed(10, _notebook_n_turns)),
    ('kicker_models',
     '[fieldmap]', str(_notebook_kicker_models), '—', ''),
    # Apertures (from config)
    ('aperture_x (±)',
     f'{_default_cfg.aperture_x_m*1e3:.1f}',
     f'{_default_cfg.aperture_x_m*1e3:.1f}', 'mm', ''),
    ('aperture_y (±)',
     f'{_default_cfg.aperture_y_m*1e3:.1f}',
     f'{_default_cfg.aperture_y_m*1e3:.1f}', 'mm', ''),
    # Injected beam
    ('injected beta_x',
     '(user defined)', _notebook_inj_beta_x, 'm', ''),
    ('injected beta_y',
     '(user defined)', _notebook_inj_beta_y, 'm', ''),
    ('injected emit_x',
     '(user defined)', f'{_notebook_inj_emit_x:.0e}', 'm·rad', ''),
    ('injected emit_y',
     '(user defined)', f'{_notebook_inj_emit_y:.0e}', 'm·rad', ''),
    ('injected x_offset',
     f'{_default_cfg.septum_x_offset_m*1e3:.1f}',
     f'{_notebook_inj_x_offset*1e3:.1f}', 'mm', ''),
    # Stored beam
    ('stored beta_x',
     '(user defined)', _notebook_sto_beta_x, 'm', ''),
    ('stored beta_y',
     '(user defined)', _notebook_sto_beta_y, 'm', ''),
    ('stored emit_x',
     '(user defined)', f'{_notebook_sto_emit_x:.0e}', 'm·rad', ''),
    ('stored emit_y',
     '(user defined)', f'{_notebook_sto_emit_y:.0e}', 'm·rad', ''),
    ('stored x_offset',
     '(user defined)', f'{_notebook_sto_x_offset*1e3:.1f}', 'mm', ''),
    # Reproducibility
    ('random seed', 42, _notebook_seed, '—',
     _changed(42, _notebook_seed)),
]

col_w = [36, 19, 19, 11, 22]
sep   = '+' + '+'.join('-' * w for w in col_w) + '+'
head  = sep
print()
print('  SIMULATION CONFIGURATION — 02_multiturn_injection_validation')
print(sep)
for i, row in enumerate(rows):
    line = '|' + '|'.join(f' {str(v):<{col_w[j]-2}} ' for j, v in enumerate(row)) + '|'
    print(line)
    if i in (0, 1):
        print(sep)
print(sep)
print()


## 1. Load Storage Ring Lattice and Kick Map

In [ ]:
config = StorageRingInjectionConfig()
ring, nkm_idx = load_storage_ring_injection_lattice(config, mat_path=repo_root / config.mat_filename)
print(f"Loaded lattice: {len(ring)} elements, NKM inserted at index {nkm_idx}")

kickmap_obj = NKMKickMap2D(repo_root / "kickmap_file.txt")
print("Loaded 2D RADIA kick map.")

## 2. Multi-Turn Injection Model Comparison

In [ ]:
n_particles = 100
n_turns = 10

injected_beam = generate_6d_beam(
    n_particles=n_particles,
    beta_x=10.0, alpha_x=0.0, emit_x=1e-7,
    beta_y=5.0, alpha_y=0.0, emit_y=1e-8,
    x_offset=-0.016,
    seed=42
)

stored_beam = generate_6d_beam(
    n_particles=n_particles,
    beta_x=10.0, alpha_x=0.0, emit_x=1e-8,
    beta_y=5.0, alpha_y=0.0, emit_y=1e-9,
    x_offset=0.0,
    seed=42
)

models = ["off", "ideal", "linear", "fieldmap"]
results = {}
inj_results_raw = {}
sto_results_raw = {}

for model in models:
    inj_res = track_multiturn_injection(injected_beam, ring, n_turns=n_turns, kicker_model=model, kickmap_obj=kickmap_obj, config=config)
    stored_res = track_multiturn_injection(stored_beam, ring, n_turns=n_turns, kicker_model=model, kickmap_obj=kickmap_obj, config=config)
    metrics = compute_multiturn_injection_metrics(inj_res, stored_res, config)
    results[model] = metrics
    inj_results_raw[model] = inj_res
    sto_results_raw[model] = stored_res
    print(f"Model [{model:8s}]: Capture = {metrics['capture_efficiency']*100:.1f}%, Stored Osc = {metrics['stored_beam_centroid_oscillation_mm']:.4f} mm")

## 3. Visualization

### 3.1 Capture Efficiency & Stored-Beam Perturbation Comparison

In [ ]:
MODEL_COLORS = {
    'off':      '#4C72B0',
    'ideal':    '#55A868',
    'linear':   '#C44E52',
    'fieldmap': '#DD8452',
}
MODEL_LABELS = {
    'off':      'NKM Off',
    'ideal':    'Ideal Kicker',
    'linear':   'Linearized NKM',
    'fieldmap': 'RADIA Fieldmap',
}

capture_pct   = [results[m]['capture_efficiency'] * 100 for m in models]
stored_osc_mm = [results[m]['stored_beam_centroid_oscillation_mm'] for m in models]
colors        = [MODEL_COLORS[m] for m in models]
labels        = [MODEL_LABELS[m] for m in models]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Multi-Turn Injection: Model Comparison', fontsize=14, fontweight='bold')

# --- Capture efficiency ---
bars = axes[0].bar(labels, capture_pct, color=colors, edgecolor='black', linewidth=0.7)
axes[0].set_ylabel('Capture Efficiency [%]')
axes[0].set_title('Injected Beam Capture Efficiency')
axes[0].set_ylim(0, 110)
axes[0].grid(axis='y', linestyle=':', alpha=0.6)
for bar, val in zip(bars, capture_pct):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

# --- Stored-beam centroid oscillation ---
bars2 = axes[1].bar(labels, stored_osc_mm, color=colors, edgecolor='black', linewidth=0.7)
axes[1].set_ylabel('Centroid Oscillation [mm]')
axes[1].set_title('Stored-Beam Centroid Perturbation')
axes[1].grid(axis='y', linestyle=':', alpha=0.6)
for bar, val in zip(bars2, stored_osc_mm):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

### 3.2 Injected Beam Phase-Space Portraits (x–x', y–y') per Model

In [ ]:
fig, axes = plt.subplots(2, len(models), figsize=(14, 8))
fig.suptitle('Injected Beam Phase-Space Portraits After Tracking', fontsize=13, fontweight='bold')

for col, model in enumerate(models):
    # particles_6d shape: (6, N)
    beam = inj_results_raw[model].particles_6d
    x_mm   = beam[0, :] * 1e3   # mm
    xp_mrad = beam[1, :] * 1e3  # mrad
    y_mm   = beam[2, :] * 1e3   # mm
    yp_mrad = beam[3, :] * 1e3  # mrad

    c = MODEL_COLORS[model]

    # Horizontal phase space
    axes[0, col].scatter(x_mm, xp_mrad, s=8, alpha=0.6, color=c, edgecolors='none')
    axes[0, col].set_title(MODEL_LABELS[model], fontsize=10, fontweight='bold')
    axes[0, col].set_xlabel('x [mm]')
    if col == 0:
        axes[0, col].set_ylabel("x' [mrad]")
    axes[0, col].axhline(0, color='k', linewidth=0.5, linestyle='--')
    axes[0, col].axvline(0, color='k', linewidth=0.5, linestyle='--')
    axes[0, col].grid(True, linestyle=':', alpha=0.4)

    # Vertical phase space
    axes[1, col].scatter(y_mm, yp_mrad, s=8, alpha=0.6, color=c, edgecolors='none')
    axes[1, col].set_xlabel('y [mm]')
    if col == 0:
        axes[1, col].set_ylabel("y' [mrad]")
    axes[1, col].axhline(0, color='k', linewidth=0.5, linestyle='--')
    axes[1, col].axvline(0, color='k', linewidth=0.5, linestyle='--')
    axes[1, col].grid(True, linestyle=':', alpha=0.4)

axes[0, 0].text(-0.25, 0.5, 'Horizontal\nPhase Space', transform=axes[0, 0].transAxes,
                rotation=90, va='center', ha='center', fontsize=10, color='gray')
axes[1, 0].text(-0.25, 0.5, 'Vertical\nPhase Space', transform=axes[1, 0].transAxes,
                rotation=90, va='center', ha='center', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

### 3.3 Survival Turn-by-Turn History (if available)

In [ ]:
# Plot survival_history if TrackingResult includes it
fig, ax = plt.subplots(figsize=(10, 4))
ax.set_title('Injected Beam: Turn-by-Turn Survival Fraction', fontsize=13, fontweight='bold')
has_history = False

for model in models:
    res = inj_results_raw[model]
    if res.survival_history is not None and len(res.survival_history) > 0:
        turns = np.arange(1, len(res.survival_history) + 1)
        surv_frac = np.array(res.survival_history) / n_particles
        ax.plot(turns, surv_frac * 100, marker='o', markersize=5,
                label=MODEL_LABELS[model], color=MODEL_COLORS[model])
        has_history = True

if has_history:
    ax.set_xlabel('Turn Number')
    ax.set_ylabel('Surviving Particles [%]')
    ax.set_ylim(0, 110)
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.6)
else:
    ax.text(0.5, 0.5,
            'survival_history not populated by this tracking backend.\n'
            'Use production tier with n_turns tracking to generate per-turn data.',
            transform=ax.transAxes, ha='center', va='center',
            fontsize=11, color='gray', style='italic')
    ax.set_axis_off()

plt.tight_layout()
plt.show()

### 3.4 Real-Space Beam Footprint: Injected vs. Stored Beam

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(14, 4), sharey=True)
fig.suptitle('Real-Space Beam Footprint at Injection Point (x–y)', fontsize=13, fontweight='bold')

for col, model in enumerate(models):
    inj_b  = inj_results_raw[model].particles_6d
    sto_b  = sto_results_raw[model].particles_6d
    config_obj = config

    ax = axes[col]
    ax.scatter(inj_b[0, :] * 1e3, inj_b[2, :] * 1e3, s=8, alpha=0.5,
               color=MODEL_COLORS[model], label='Injected', edgecolors='none')
    ax.scatter(sto_b[0, :] * 1e3, sto_b[2, :] * 1e3, s=8, alpha=0.35,
               color='gray', label='Stored', edgecolors='none')

    # Draw aperture
    apx = config_obj.aperture_x_m * 1e3
    apy = config_obj.aperture_y_m * 1e3
    rect = plt.Rectangle((-apx, -apy), 2*apx, 2*apy,
                          fill=False, edgecolor='red', linewidth=1.5, linestyle='--', label='Aperture (±30/±15 mm)')
    ax.add_patch(rect)

    ax.set_title(MODEL_LABELS[model], fontsize=10, fontweight='bold')
    ax.set_xlabel('x [mm]')
    if col == 0:
        ax.set_ylabel('y [mm]')
    ax.set_xlim(-50, 20)
    ax.set_ylim(-20, 20)
    ax.grid(True, linestyle=':', alpha=0.4)
    if col == 0:
        ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

### 3.5 Injection Summary Metrics Table

In [ ]:
print()
print(f"{'Model':<16} {'Capture [%]':>12} {'Inj Emit-x [nm·rad]':>21} {'Stored Osc [mm]':>16} {'Inj Survive':>12}")
print('-' * 82)
for model in models:
    m = results[model]
    inj_r = inj_results_raw[model]
    print(f"{MODEL_LABELS[model]:<16}"
          f" {m['capture_efficiency']*100:>12.1f}"
          f" {inj_r.emittance_x_mrad * 1e6:>21.4f}"
          f" {m['stored_beam_centroid_oscillation_mm']:>16.4f}"
          f" {inj_r.survived_particles:>12d}")
print()